真实的和预测的细胞掩码 target_cell_id/pred_cell_id、细胞核掩码 input_nuclei，以及真实的和预测的表达谱 target_expr/pred_expr

Key: input_nuclei    | Shape: [512, 512, 1]             | Dtype: torch.int32     | Layout: torch.sparse_coo

Key: target_expr     | Shape: [512, 512, 1696]          | Dtype: torch.float32   | Layout: torch.sparse_coo

Key: target_cell_id  | Shape: [512, 512, 1]             | Dtype: torch.int32     | Layout: torch.sparse_coo

Key: pred_expr       | Shape: [1696, 512, 512]          | Dtype: torch.float32   | Layout: torch.sparse_coo

Key: pred_cell_id    | Shape: [1, 512, 512]             | Dtype: torch.float32   | Layout: torch.

对应的值必须全部为 PyTorch 稀疏张量 (Sparse Tensor) 格式
如果数据仍然为密集矩阵,可以使用convert_dense_to_sparse函数来完成转换。

In [ ]:
import os
import glob
import torch

# 将密集矩阵数据转换为稀疏矩阵数据文件, 便于后续进行评估和转换为CSR矩阵返还anndata数据
def convert_dense_to_sparse(input_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    file_paths = glob.glob(os.path.join(input_dir, "*.pt"))
    
    keys_to_keep = [
        "target_expr", 
        "target_cell_id", 
        "pred_expr", 
        "pred_cell_id"
    ]
    
    for file_path in file_paths:
        dense_data = torch.load(file_path, weights_only=True)
        sparse_data = {}
        
        for key in keys_to_keep:
            # 提取指定键并直接转换为稀疏张量
            sparse_data[key] = dense_data[key].to_sparse()
            
        filename = os.path.basename(file_path)
        save_path = os.path.join(output_dir, filename)
        
        torch.save(sparse_data, save_path)

if __name__ == "__main__":
    INPUT_DIR = "/root/autodl-tmp/dense_test_samples"
    OUTPUT_DIR = "/root/autodl-tmp/sparse_test_samples"
    
    convert_dense_to_sparse(INPUT_DIR, OUTPUT_DIR)

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from scipy.special import comb
from glob import glob
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 辅助函数：稀疏张量转 DataFrame
# ==========================================
def sparse_to_df(sparse_tensor, val_col_name, W):
    """将 [1, H, W] 的稀疏张量转换为包含 1D 索引和值的 DataFrame"""
    ind = sparse_tensor.indices()
    val = sparse_tensor.values().numpy()
    
    # 展平空间坐标为 1D 索引: idx = y * W + x
    idx_1d = (ind[1] * W + ind[2]).numpy()
    
    # 过滤掉背景 (0)
    mask = val > 0
    return pd.DataFrame({'idx': idx_1d[mask], val_col_name: val[mask]})

# ==========================================
# 模块一：极速形态学分割指标计算 (基于 Pandas 向量化)
# ==========================================
def evaluate_segmentation_sparse(df_p, df_t, df_n, H, W):
    # 1. 语义级基础数据 (交集与并集)
    intersection = np.intersect1d(df_p['idx'], df_t['idx']).size
    pred_fg_sum = len(df_p)
    target_fg_sum = len(df_t)
    union = pred_fg_sum + target_fg_sum - intersection
    
    iou = intersection / union if union > 0 else 0.0
    dice = 2 * intersection / (pred_fg_sum + target_fg_sum) if (pred_fg_sum + target_fg_sum) > 0 else 0.0
    
    # 2. 实例级指标 (极速 ARI 计算)
    # 使用外连接获取所有前景像素
    df_all = pd.merge(df_p, df_t, on='idx', how='outer').fillna(0)
    bg_bg_count = H * W - len(df_all) # 纯背景像素数
    
    if len(df_all) > 0:
        pair_counts = df_all.groupby(['t_id', 'p_id']).size().values
        a_i = df_all['t_id'].value_counts().to_dict()
        b_j = df_all['p_id'].value_counts().to_dict()
        
        a_i[0] = a_i.get(0, 0) + bg_bg_count
        b_j[0] = b_j.get(0, 0) + bg_bg_count
        
        sum_comb_n_ij = sum(comb(c, 2, exact=False) for c in pair_counts) + comb(bg_bg_count, 2, exact=False)
        sum_comb_a_i = sum(comb(v, 2, exact=False) for v in a_i.values())
        sum_comb_b_j = sum(comb(v, 2, exact=False) for v in b_j.values())
        comb_n = comb(H * W, 2, exact=False)
        
        expected_index = (sum_comb_a_i * sum_comb_b_j) / comb_n
        max_index = (sum_comb_a_i + sum_comb_b_j) / 2
        ari = (sum_comb_n_ij - expected_index) / (max_index - expected_index) if max_index != expected_index else 1.0
    else:
        ari = 1.0

    # 3. 细胞数量统计
    gt_cells = df_t['t_id'].nunique()
    pred_cells = df_p['p_id'].nunique()
    
    # 4. 无核细胞统计
    overlapping_gt_ids = df_t[df_t['idx'].isin(df_n['idx'])]['t_id'].unique()
    all_gt_ids = df_t['t_id'].unique()
    gt_non_nuc_ids = np.setdiff1d(all_gt_ids, overlapping_gt_ids)
    
    overlapping_pred_ids = df_p[df_p['idx'].isin(df_n['idx'])]['p_id'].unique()
    all_pred_ids = df_p['p_id'].unique()
    pred_non_nuc_ids = np.setdiff1d(all_pred_ids, overlapping_pred_ids)
    
    TP = 0
    df_inter = pd.merge(df_p, df_t, on='idx', how='inner')
    df_inter_nn = df_inter[df_inter['p_id'].isin(pred_non_nuc_ids)]
    
    if not df_inter_nn.empty:
        # 统计每个 (p_id, t_id) 对的交集大小
        pair_intersections = df_inter_nn.groupby(['p_id', 't_id']).size().reset_index(name='intersection')
        # 找到每个 p_id 对应的最大交集 t_id (相当于 mode)
        best_matches = pair_intersections.sort_values('intersection', ascending=False).drop_duplicates('p_id')
        # 过滤出 mode 属于 gt_non_nuc_ids 的匹配
        valid_matches = best_matches[best_matches['t_id'].isin(gt_non_nuc_ids)]
        
        p_areas = df_p['p_id'].value_counts().to_dict()
        t_areas = df_t['t_id'].value_counts().to_dict()
        
        for _, row in valid_matches.iterrows():
            p_id, t_id, inter = row['p_id'], row['t_id'], row['intersection']
            inst_union = p_areas[p_id] + t_areas[t_id] - inter
            if (inter / inst_union) > 0.5:
                TP += 1
                
    gt_non_nuc_count = len(gt_non_nuc_ids)
    pred_non_nuc_count = len(pred_non_nuc_ids)
    
    recall = TP / gt_non_nuc_count if gt_non_nuc_count > 0 else np.nan
    precision = TP / pred_non_nuc_count if pred_non_nuc_count > 0 else np.nan

    return {
        "IoU": iou, "Dice": dice, "ARI": ari,
        "NonNuc_Recall": recall, "NonNuc_Precision": precision,
        "intersection": intersection, "union": union,
        "pred_fg_sum": pred_fg_sum, "target_fg_sum": target_fg_sum,
        "gt_cells": gt_cells, "pred_cells": pred_cells,
        "gt_non_nuc": gt_non_nuc_count, "pred_non_nuc": pred_non_nuc_count,
        "non_nuc_tp": TP
    }

# ==========================================
# 模块二：极速表达谱指标计算 (基于 1D 索引映射)
# ==========================================
def extract_compact_dense_expr(sparse_expr, lookup_table, C, N, W):
    """利用查找表，仅提取有效像素，构建紧凑的 [C, N] 密集矩阵"""
    ind = sparse_expr.indices()
    val = sparse_expr.values()
    
    c = ind[0]
    idx_1d = ind[1] * W + ind[2]
    
    # 映射到 0~N-1 的列索引
    mapped_cols = lookup_table[idx_1d]
    valid_mask = mapped_cols >= 0
    
    c_valid = c[valid_mask]
    cols_valid = mapped_cols[valid_mask]
    vals_valid = val[valid_mask]
    
    # 构建紧凑矩阵
    dense_mat = torch.zeros((C, N), dtype=torch.float32)
    dense_mat[c_valid, cols_valid] = vals_valid.float()
    return dense_mat

def evaluate_expression_sparse(pred_expr, target_expr, valid_idx_array, H, W, max_samples=500000):
    N = len(valid_idx_array)
    if N == 0:
        return {"RMSE": np.nan, "PCC": np.nan, "SCC": np.nan, "sse": 0.0, "valid_bins_count": 0}
    
    C = target_expr.shape[0]
    
    # 构建 1D 查找表 (大小为 H*W，非常省内存且极速)
    lookup = torch.full((H * W,), -1, dtype=torch.long)
    lookup[torch.from_numpy(valid_idx_array)] = torch.arange(N)
    
    # 提取紧凑的 [C, N] 矩阵
    t_expr_dense = extract_compact_dense_expr(target_expr, lookup, C, N, W)
    p_expr_dense = extract_compact_dense_expr(pred_expr, lookup, C, N, W)
    
    t_flat = t_expr_dense.flatten().numpy()
    p_flat = p_expr_dense.flatten().numpy()
    
    valid_bins_count = len(t_flat)
    sse = np.sum((p_flat - t_flat) ** 2)
    rmse = np.sqrt(sse / valid_bins_count) if valid_bins_count > 0 else np.nan
    
    if valid_bins_count > max_samples:
        indices = np.random.choice(valid_bins_count, max_samples, replace=False)
        p_sample = p_flat[indices]
        t_sample = t_flat[indices]
    else:
        p_sample = p_flat
        t_sample = t_flat
        
    if np.std(p_sample) == 0 or np.std(t_sample) == 0:
        pcc, scc = 0.0, 0.0
    else:
        pcc = np.corrcoef(p_sample, t_sample)[0, 1]
        scc, _ = spearmanr(p_sample, t_sample)
    
    return {"RMSE": rmse, "PCC": pcc, "SCC": scc, "sse": sse, "valid_bins_count": valid_bins_count}

# ==========================================
# 主函数
# ==========================================
def main():
    data_dir = "/root/autodl-tmp/data_mocked"
    pt_files = glob(os.path.join(data_dir, "*.pt"))
    
    if not pt_files:
        print(f"未在 {data_dir} 找到任何 .pt 文件！")
        return

    print(f"找到 {len(pt_files)} 个测试文件，开始评估:")

    file_metrics = []
    global_stats = {
        "intersection": 0, "union": 0, "pred_fg_sum": 0, "target_fg_sum": 0,
        "sse": 0.0, "valid_bins_count": 0, "ari_list": [], "pcc_list": [], "scc_list": [],
        "non_nuc_tp": 0, "gt_non_nuc_total": 0, "pred_non_nuc_total": 0
    }

    for filepath in tqdm(pt_files, desc="Evaluating"):
        filename = os.path.basename(filepath)
        data = torch.load(filepath)
        
        # 获取图像尺寸
        _, H, W = data['target_cell_id'].shape
        
        # 1. 将稀疏掩码转换为 DataFrame (极速向量化准备)
        df_t = sparse_to_df(data['target_cell_id'], 't_id', W)
        df_p = sparse_to_df(data['pred_cell_id'], 'p_id', W)
        df_n = sparse_to_df(data['input_nuclei'], 'n_id', W)
        
        # 2. 计算形态学指标
        seg_res = evaluate_segmentation_sparse(df_p, df_t, df_n, H, W)
        
        # 3. 计算表达谱指标 (仅在 target_cell_id > 0 的有效像素上计算)
        valid_idx_array = df_t['idx'].values
        expr_res = evaluate_expression_sparse(data['pred_expr'], data['target_expr'], valid_idx_array, H, W)
        
        # 4. 记录单图结果
        file_metrics.append({
            "Filename": filename,
            "IoU": seg_res["IoU"], "Dice": seg_res["Dice"], "ARI": seg_res["ARI"],
            "NonNuc_Recall": seg_res["NonNuc_Recall"], "NonNuc_Precision": seg_res["NonNuc_Precision"],
            "RMSE": expr_res["RMSE"], "PCC": expr_res["PCC"], "SCC": expr_res["SCC"]
        })
        
        # 5. 累加全局统计量
        global_stats["intersection"] += seg_res["intersection"]
        global_stats["union"] += seg_res["union"]
        global_stats["pred_fg_sum"] += seg_res["pred_fg_sum"]
        global_stats["target_fg_sum"] += seg_res["target_fg_sum"]
        global_stats["sse"] += expr_res["sse"]
        global_stats["valid_bins_count"] += expr_res["valid_bins_count"]
        global_stats["non_nuc_tp"] += seg_res["non_nuc_tp"]
        global_stats["gt_non_nuc_total"] += seg_res["gt_non_nuc"]
        global_stats["pred_non_nuc_total"] += seg_res["pred_non_nuc"]
        
        if not np.isnan(seg_res["ARI"]): global_stats["ari_list"].append(seg_res["ARI"])
        if not np.isnan(expr_res["PCC"]): global_stats["pcc_list"].append(expr_res["PCC"])
        if not np.isnan(expr_res["SCC"]): global_stats["scc_list"].append(expr_res["SCC"])

    # 计算并输出全局指标
    global_iou = global_stats["intersection"] / global_stats["union"] if global_stats["union"] > 0 else 0
    fg_sum_total = global_stats["pred_fg_sum"] + global_stats["target_fg_sum"]
    global_dice = 2 * global_stats["intersection"] / fg_sum_total if fg_sum_total > 0 else 0
    global_rmse = np.sqrt(global_stats["sse"] / global_stats["valid_bins_count"]) if global_stats["valid_bins_count"] > 0 else np.nan
    global_nonnuc_recall = global_stats["non_nuc_tp"] / global_stats["gt_non_nuc_total"] if global_stats["gt_non_nuc_total"] > 0 else np.nan
    global_nonnuc_precision = global_stats["non_nuc_tp"] / global_stats["pred_non_nuc_total"] if global_stats["pred_non_nuc_total"] > 0 else np.nan

    mean_ari = np.mean(global_stats["ari_list"]) if global_stats["ari_list"] else np.nan
    mean_pcc = np.mean(global_stats["pcc_list"]) if global_stats["pcc_list"] else np.nan
    mean_scc = np.mean(global_stats["scc_list"]) if global_stats["scc_list"] else np.nan

    print("\n" + "="*45)
    print("全局评估结果 (Global Metrics)")
    print("="*45)
    print(f"Global IoU             : {global_iou:.4f}")
    print(f"Global Dice            : {global_dice:.4f}")
    print(f"Mean ARI               : {mean_ari:.4f}")
    print(f"Global NonNuc Recall   : {global_nonnuc_recall:.4f}")
    print(f"Global NonNuc Precision: {global_nonnuc_precision:.4f}")
    print("-" * 45)
    print(f"Global RMSE            : {global_rmse:.4f}")
    print(f"Mean PCC               : {mean_pcc:.4f}")
    print(f"Mean SCC               : {mean_scc:.4f}")
    print("="*45)

    # 保存 CSV
    df_metrics = pd.DataFrame(file_metrics)
    csv_path = os.path.join(data_dir, "evaluation_results.csv")
    df_metrics.to_csv(csv_path, index=False)
    print(f"\n 详细的单图评估结果已保存至: {csv_path}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
import anndata as ad

def reconstruct_anndata_from_tiles(data_dir, target_key='pred_expr'):
    """
    从切片目录中重组 AnnData 数据
    :param data_dir: .pt 文件所在目录
    :param target_key: 需要提取的表达谱键名, 如 'target_expr' 或 'pred_expr'
    :return: 重组后的 AnnData 对象
    """
    all_y, all_x, all_c, all_v = [], [], [], []
    
    print("1. 正在读取并转换切片坐标...")
    for fname in os.listdir(data_dir):
        if not fname.endswith('.pt'):
            continue
            
        # 从文件名解析全局起始坐标 y0, x0
        # 文件名格式: tile_{y0}_{x0}.pt
        parts = fname.replace('.pt', '').split('_')
        y0, x0 = int(parts[1]), int(parts[2])
        
        # 加载数据
        file_path = os.path.join(data_dir, fname)
        tile_data = torch.load(file_path)
        
        if target_key not in tile_data:
            continue
            
        # 获取稀疏张量并确保其为 coalesce 状态
        expr_tensor = tile_data[target_key].coalesce()
        indices = expr_tensor.indices().numpy()  # 形状: [3, N], 分别为 [y_local, x_local, c]
        values = expr_tensor.values().numpy()    # 形状: [N]
        
        # 局部坐标转换为全局坐标
        y_global = indices[0] + y0
        x_global = indices[1] + x0
        c = indices[2]
        
        all_y.append(y_global)
        all_x.append(x_global)
        all_c.append(c)
        all_v.append(values)
        
    if not all_y:
        raise ValueError("未找到有效数据，请检查目录或 target_key 是否正确。")

    # 拼接所有切片的数据
    all_y = np.concatenate(all_y)
    all_x = np.concatenate(all_x)
    all_c = np.concatenate(all_c)
    all_v = np.concatenate(all_v)
    
    print("2. 正在处理 Overlap 导致的重复数据...")
    # 使用 Pandas 快速去重。因为 overlap 区域的坐标和表达量完全一致，直接按坐标去重即可
    df = pd.DataFrame({'y': all_y, 'x': all_x, 'c': all_c, 'v': all_v})
    df = df.drop_duplicates(subset=['y', 'x', 'c'])
    
    print("3. 正在构建 AnnData CSR 矩阵...")
    # 提取所有有表达的唯一空间坐标点 (像素/Spot)
    spatial_coords = df[['y', 'x']].drop_duplicates().reset_index(drop=True)
    spatial_coords['obs_id'] = spatial_coords.index  # 为每个空间点分配一个行索引
    
    # 将行索引映射回主数据表
    df = df.merge(spatial_coords, on=['y', 'x'])
    
    row_indices = df['obs_id'].values
    col_indices = df['c'].values
    data_values = df['v'].values
    
    num_obs = len(spatial_coords)
    num_vars = all_c.max() + 1 if len(all_c) > 0 else 0
    
    # 构建 COO 矩阵并转换为 CSR 矩阵
    sparse_mat = coo_matrix(
        (data_values, (row_indices, col_indices)), 
        shape=(num_obs, num_vars)
    ).tocsr()
    
    # 构建 AnnData
    adata = ad.AnnData(X=sparse_mat)
    
    # 保存空间坐标 (通常习惯保存为 [x, y] 格式)
    adata.obsm['spatial'] = spatial_coords[['x', 'y']].values
    
    print(f"重组完成！AnnData 包含 {num_obs} 个空间位点，{num_vars} 个基因通道。")
    return adata

In [ ]:
if __name__ == "__main__":
    data_dir = "/root/autodl-tmp/data_mocked"
    # 重组真实的表达谱
    adata_target = reconstruct_anndata_from_tiles(data_dir, target_key='target_expr')
    print(adata_target)